In [1]:
import os
from typing import List, Tuple, Callable, TypeAlias, OrderedDict, Dict, Any, Sequence
from pathlib import Path
from dataclasses import dataclass
import warnings


import onnx
from onnxconverter_common import float16
import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn

from timm.models.resnet import resnet18, resnet34, resnet50, resnet101, resnet152
from timm.models.convnext import (
    convnext_base,
    convnext_small,
    convnext_tiny,
)
from timm.models.efficientnet import (
    efficientnet_b0,
    efficientnet_b1,
    efficientnet_b2,
    efficientnet_b3,
    efficientnet_b4,
    efficientnet_b5,
    efficientnet_b6,
    efficientnet_b7,
    efficientnetv2_s,
    efficientnetv2_m,
)
from timm.models.swin_transformer import (
    swin_tiny_patch4_window7_224,
    swin_small_patch4_window7_224,
    swin_base_patch4_window7_224,
)
from timm.models.vision_transformer import vit_base_patch16_224
from timm.models.mobilenetv3 import (
    mobilenetv4_conv_medium,
    mobilenetv4_conv_small,
    mobilenetv4_hybrid_medium,
    mobilenetv4_conv_large,
    mobilenetv4_hybrid_large,
)
from timm.models.mobilevit import (
    mobilevitv2_050,
    mobilevitv2_075,
    mobilevitv2_100,
    mobilevitv2_125,
    mobilevitv2_150,
    mobilevitv2_175,
    mobilevitv2_200,
)

ARCHS = {
    "resnet_18": resnet18,
    "resnet_34": resnet34,
    "resnet_50": resnet50,
    "resnet_101": resnet101,
    "resnet_152": resnet152,
    "efficientnet_b0": efficientnet_b0,
    "efficientnet_b1": efficientnet_b1,
    "efficientnet_b2": efficientnet_b2,
    "efficientnet_b3": efficientnet_b3,
    "efficientnet_b4": efficientnet_b4,
    "efficientnet_b5": efficientnet_b5,
    "efficientnet_b6": efficientnet_b6,
    "efficientnet_b7": efficientnet_b7,
    "efficientnetv2_s": efficientnetv2_s,
    "efficientnetv2_m": efficientnetv2_m,
    "swin_tiny": swin_tiny_patch4_window7_224,
    "swin_small": swin_small_patch4_window7_224,
    "swin_base": swin_base_patch4_window7_224,
    "convnext_tiny": convnext_tiny,
    "convnext_small": convnext_small,
    "convnext_base": convnext_base,
    "vit_b": vit_base_patch16_224,
    "mobilevit_050": mobilevitv2_050,
    "mobilevit_075": mobilevitv2_075,
    "mobilevit_100": mobilevitv2_100,
    "mobilevit_125": mobilevitv2_125,
    "mobilevit_150": mobilevitv2_150,
    "mobilevit_175": mobilevitv2_175,
    "mobilevit_200": mobilevitv2_200,
    "mobilenet_conv_small": mobilenetv4_conv_small,
    "mobilenet_conv_medium": mobilenetv4_conv_medium,
    "mobilenet_hybrid_medium": mobilenetv4_hybrid_medium,
    "mobilenet_conv_large": mobilenetv4_conv_large,
    "mobilenet_hybrid_large": mobilenetv4_hybrid_large,
}

IN_CHANNELS = 3
NUM_CLASSES = 656
DEVICE = "cuda:1"
# DEVICE = "cpu"


def load_ckpt(ckpt_path: str) -> OrderedDict:
    state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"]
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        newk: str = k.replace("model.", "")  # lightning module
        newk = newk.replace("_orig_mod.", "")  # torch.compile OptimizedModule
        new_state_dict[newk] = v
    return new_state_dict


def load_model(
    name: str,
    ckpt_path: str = "",
    in_channels: int = 3,
    num_classes: int = 656,
    compile: bool = False,
) -> torch.nn.Module | Any:
    func: Callable[..., nn.Module] = ARCHS.get(name, None)  # type: ignore
    assert func is not None, f"Unknown model name: {name}"
    model = func(in_chans=in_channels, num_classes=num_classes)
    if os.path.exists(ckpt_path):
        state_dict = load_ckpt(ckpt_path)
        model.load_state_dict(state_dict)
    if compile:
        model = torch.compile(model)
    return model


def convert_onnx(
    model: nn.Module,
    onnx_path: str,
    input_shape: Tuple[int, int, int, int] = (1, NUM_CLASSES, 224, 224),
    device: str = "cpu",
):
    model.eval()
    model = model.to(device)
    dummy_input = torch.randn(input_shape, device=device, dtype=torch.float32)

    with torch.no_grad():
        output_true = model(dummy_input)

    torch.onnx.export(
        model,
        (dummy_input,),
        onnx_path,
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {0: "batch_size"},
        },
    )

    onnx_model = onnx.load(onnx_path)
    model_fp16 = float16.convert_float_to_float16(onnx_model)
    onnx_path_bf16 = (
        Path(onnx_path).parent / "f16" / Path(onnx_path).name.replace(".onnx", "_f16.onnx")
    )
    onnx.save(model_fp16, onnx_path_bf16)
    print(f"Exported ONNX model to {onnx_path} and {onnx_path_bf16}")

    onnx_model = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    onnx_inputs = {onnx_model.get_inputs()[0].name: dummy_input.cpu().numpy()}
    output_onnx = onnx_model.run(None, onnx_inputs)[0]

    assert np.allclose(output_true.cpu().numpy(), output_onnx, atol=1e-4), (
        "ONNX model output is not correct"
    )

    print("ONNX model is correct")


In [ ]:
@dataclass
class ExportConfig:
    arch: str
    ckpt_path: str
    seed: int
    input_shape: Tuple[int, int, int, int]


CKPT_PARENT_DIR = Path("../logs/repeat/pretrained_False")

export_configs = [
    ExportConfig(
        arch="mobilenet_conv_small",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet/conv_small/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_conv_medium",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet/conv_medium/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_hybrid_medium",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet/hybrid_medium/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_conv_large",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet/conv_large/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_hybrid_large",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet/hybrid_large/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="resnet_18",
        ckpt_path=f"{CKPT_PARENT_DIR}/resnet/18/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(4, 3, 224, 224),
    ),
    ExportConfig(
        arch="resnet_34",
        ckpt_path=f"{CKPT_PARENT_DIR}/resnet/34/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(4, 3, 224, 224),
    ),
    ExportConfig(
        arch="resnet_50",
        ckpt_path=f"{CKPT_PARENT_DIR}/resnet/50/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="swin_tiny",
        ckpt_path=f"{CKPT_PARENT_DIR}/swin/tiny/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="convnext_tiny",
        ckpt_path=f"{CKPT_PARENT_DIR}/convnext/tiny/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilevit_050",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilevit/050/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilevit_100",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilevit/100/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilevit_150",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilevit/150/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="vit_b",
        ckpt_path=f"{CKPT_PARENT_DIR}/vit/b/42/checkpoints/last.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
]

with warnings.catch_warnings(action="ignore"):
    for config in export_configs:
        model = load_model(config.arch, config.ckpt_path, IN_CHANNELS, NUM_CLASSES, False)
        convert_onnx(
            model,
            f"export_models/{config.arch}_{config.seed}.onnx",
            input_shape=config.input_shape,
        )

Exported ONNX model to export_models/mobilenet_conv_small_42.onnx and export_models/mobilenet_conv_small_42_f16.onnx
ONNX model is correct
Exported ONNX model to export_models/mobilenet_conv_medium_42.onnx and export_models/mobilenet_conv_medium_42_f16.onnx
ONNX model is correct
Exported ONNX model to export_models/mobilenet_hybrid_medium_42.onnx and export_models/mobilenet_hybrid_medium_42_f16.onnx
ONNX model is correct
Exported ONNX model to export_models/mobilenet_conv_large_42.onnx and export_models/mobilenet_conv_large_42_f16.onnx
ONNX model is correct
Exported ONNX model to export_models/mobilenet_hybrid_large_42.onnx and export_models/mobilenet_hybrid_large_42_f16.onnx
ONNX model is correct
Exported ONNX model to export_models/resnet_18_42.onnx and export_models/resnet_18_42_f16.onnx
ONNX model is correct
Exported ONNX model to export_models/resnet_34_42.onnx and export_models/resnet_34_42_f16.onnx
ONNX model is correct
Exported ONNX model to export_models/resnet_50_42.onnx and 

In [ ]:
for config in export_configs:
    os.system(
        "mnnconvert -f ONNX --fp16 "
        f"--modelFile export_models/{config.arch}_{config.seed}.onnx "
        f"--MNNModel export_models/mnn/{config.arch}_{config.seed}.mnn "
        f"--bizCode {config.arch}_{config.seed}"
    )


CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22  0  14  24  2  16  26  4  18  28  6  8 ], 600000 - 5752000
The device supports: i8sdot:0, fp16:0, i8mm: 0, sve2: 0, sme2: 0
CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22  0  14  24  2  16  26  4  18  28  6  8 ], 600000 - 5752000
The device supports: i8sdot:0, fp16:0, i8mm: 0, sve2: 0, sme2: 0
Start to Convert Other Model Format To MNN Model..., target version: 3.1
[17:26:22] :46: ONNX Model ir version: 8
[17:26:22] :47: ONNX Model opset version: 18
Start to Optimize the MNN Net...
inputTensors : [ input, ]
outputTensors: [ output, ]
Converted Success!
CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22  0  14  24  2  16  26  4  18  28  6  8 ], 600000 - 5752000
The device supports: i8sdot:0, fp16:0, i8mm: 0, sve2: 0, sme2: 0
CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22 

In [16]:
@dataclass
class ExportConfig:
    arch: str
    ckpt_path: str
    seed: int
    input_shape: Tuple[int, int, int, int]


CKPT_PARENT_DIR = Path("<placeholder>/OpenSeed-Models/pre_trained/torch")

export_configs = [
    ExportConfig(
        arch="mobilenet_conv_small",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet_conv_small_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_conv_medium",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet_conv_medium_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_hybrid_medium",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet_hybrid_medium_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_conv_large",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet_conv_large_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilenet_hybrid_large",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilenet_hybrid_large_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="resnet_18",
        ckpt_path=f"{CKPT_PARENT_DIR}/resnet_18_seed42.ckpt",
        seed=42,
        input_shape=(4, 3, 224, 224),
    ),
    ExportConfig(
        arch="resnet_34",
        ckpt_path=f"{CKPT_PARENT_DIR}/resnet_34_seed42.ckpt",
        seed=42,
        input_shape=(4, 3, 224, 224),
    ),
    ExportConfig(
        arch="resnet_50",
        ckpt_path=f"{CKPT_PARENT_DIR}/resnet_50_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="swin_tiny",
        ckpt_path=f"{CKPT_PARENT_DIR}/swin_tiny_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="convnext_tiny",
        ckpt_path=f"{CKPT_PARENT_DIR}/convnext_tiny_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilevit_050",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilevit_0.5_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilevit_100",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilevit_1.0_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="mobilevit_150",
        ckpt_path=f"{CKPT_PARENT_DIR}/mobilevit_1.5_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
    ExportConfig(
        arch="vit_b",
        ckpt_path=f"{CKPT_PARENT_DIR}/vit_b_42_seed42.ckpt",
        seed=42,
        input_shape=(1, 3, 224, 224),
    ),
]

# with warnings.catch_warnings(action="ignore"):
#     for config in export_configs:
#         model = load_model(config.arch, config.ckpt_path, IN_CHANNELS, NUM_CLASSES, False)
#         convert_onnx(
#             model,
#             f"<placeholder>/OpenSeed-Models/pre_trained/onnx/{config.arch}_seed{config.seed}.onnx",
#             input_shape=config.input_shape,
#         )

In [ ]:
for config in export_configs:
    os.system(
        "mnnconvert -f ONNX --fp16 "
        f"--modelFile <placeholder>/OpenSeed-Models/scratch_trained/onnx/{config.arch}_seed{config.seed}.onnx "
        f"--MNNModel <placeholder>/OpenSeed-Models/scratch_trained/mnn/{config.arch}_seed{config.seed}.mnn "
        f"--bizCode \"This model [{config.arch}_seed{config.seed}] is part of OpenSeed project created by HuLab-LZU\""
    )

CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22  0  14  24  2  16  26  4  18  28  6  8 ], 600000 - 5752000
The device supports: i8sdot:0, fp16:0, i8mm: 0, sve2: 0, sme2: 0
CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22  0  14  24  2  16  26  4  18  28  6  8 ], 600000 - 5752000
The device supports: i8sdot:0, fp16:0, i8mm: 0, sve2: 0, sme2: 0
Start to Convert Other Model Format To MNN Model..., target version: 3.2
[17:31:10] :46: ONNX Model ir version: 8
[17:31:10] :47: ONNX Model opset version: 18
Start to Optimize the MNN Net...
inputTensors : [ input, ]
outputTensors: [ output, ]
Converted Success!
CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22  0  14  24  2  16  26  4  18  28  6  8 ], 600000 - 5752000
The device supports: i8sdot:0, fp16:0, i8mm: 0, sve2: 0, sme2: 0
CPU Group: [ 20  21  31  13  23  1  15  25  3  17  27  5  19  29  7  10  11  30  9  12  22 